Categorical plots: https://seaborn.pydata.org/tutorial/categorical.html

V-Measure paper (with definition of F-Measure): https://www.aclweb.org/anthology/D07-1043.pdf

In [135]:
import json

from matplotlib.backends.backend_pdf import PdfPages
from weasyprint import HTML

from utils.analysis import align_clusterings_for_sklearn
from utils.io import load_clustering, get_review_pmids
from utils.preprocessing import preprocess_clustering, get_clustering_level

In [2]:
with open('grid_search_full_2021-03-03_13_22.json', 'r') as f:
    results = json.load(f)

In [85]:
ground_truth = {}

for pmid in get_review_pmids():
    clustering = load_clustering(pmid)
    for level in range(1, get_clustering_level(clustering)):
        ground_truth[(pmid, level)] = preprocess_clustering(clustering, max_level=level,
                                                            include_box_sections=False,
                                                            uniqueness_method='unique_only')

In [132]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="ticks", color_codes=True)

In [4]:
PARAM_NAMES = ['bibcoupling', 'cocitation', 'citation', 'text_citation']

In [81]:
from sklearn.metrics.cluster import contingency_matrix

def my_score(labels_true, labels_pred):
    contingency = contingency_matrix(labels_true, labels_pred)
    
    row_sums = contingency.sum(axis=1)
    contingency_row = contingency / row_sums[:, np.newaxis]
    col_sums = contingency.sum(axis=0)
    contingency_col = contingency / col_sums[np.newaxis, :]
    
    diversity = np.sqrt(np.diag(contingency_row @ contingency_row.T))
    purity = np.diag(contingency_row @ contingency_col.T)
    return np.mean(purity * diversity)

In [118]:
score_data = []
partitions = {}

for paper in results:
    pmid = paper['pmid']
    level = paper['level']
    full_id = f'{pmid} ({level})'
    grid_results = paper['results']
    for result in grid_results:
        soi = result['soi']
        data = result['results']
        partition = data['partition']
        score = data['v_measure_score']
        
        max_soi = max(soi)
        main_param_idx = soi.index(max_soi)
        main_param = PARAM_NAMES[main_param_idx]
        n_clusters = len(set(partition.values()))
        
        labels_true, labels_pred = align_clusterings_for_sklearn(ground_truth[(pmid, level)], partition)
        
        score_data.append((pmid, level, *soi, 
                           score, my_score(labels_true, labels_pred), 
                           n_clusters, main_param, full_id))
        partitions[(pmid, level, *soi)] = partition

In [119]:
score_df = pd.DataFrame(score_data, columns=['PMID', 'Level', *PARAM_NAMES, 
                                             'V-Score', 'My-Score',
                                             'Clusters', 'Largest Parameter', 'PMID (level)'])

In [188]:
score_df.head(5)

,PMID,Level,bibcoupling,cocitation,citation,text_citation,V-Score,My-Score,Clusters,Largest Parameter,PMID (level),V-0.01*C,V-0.005*C
0,26580716,1,0.0,1,0.0,0.0000,0.360186,0.217836,7,cocitation,26580716 (1),0.290186,0.325186
1,26580716,1,0.0,1,0.0,0.0625,0.359994,0.216823,7,cocitation,26580716 (1),0.289994,0.324994
2,26580716,1,0.0,1,0.0,0.1250,0.359994,0.216823,7,cocitation,26580716 (1),0.289994,0.324994
3,26580716,1,0.0,1,0.0,0.2500,0.361698,0.218405,7,cocitation,26580716 (1),0.291698,0.326698
4,26580716,1,0.0,1,0.0,0.5000,0.361280,0.229159,7,cocitation,26580716 (1),0.291280,0.326280


In [155]:
def get_best_clustering(score_df, target_col, unique=False):
    cols_to_select = ['PMID', 'Level', *PARAM_NAMES, target_col, 'Clusters']
    idx = score_df.groupby('PMID (level)')[target_col].transform(max) == score_df[target_col]
    if unique:
        return score_df.loc[idx, cols_to_select].drop_duplicates(subset=['PMID', 'Level'])
    return score_df.loc[idx, cols_to_select]

In [184]:
def plot_catplot(score_df, target_col, filename):
    g = sns.catplot(x=target_col, y="PMID (level)", hue="Largest Parameter", 
                    kind="strip", data=score_df, height=40, aspect=0.5)
    g.savefig(filename)

In [122]:
def add_reg_column(score_df, reg):
    reg_column = f'V-{reg}*C'
    score_df[reg_column] = score_df['V-Score'] - reg * score_df['Clusters']
    return score_df, reg_column

In [162]:
def plot_contingency_matrix(cm, title_str):
    fig = plt.figure(figsize = (10,7))
    sns.heatmap(cm, annot=True)
    plt.title(title_str)
    plt.xlabel('PubTrends')
    plt.ylabel('Nature Reviews')
    return fig

In [151]:
def print_dataframe_to_pdf(df, filename):
    table = df.to_html(classes='mystyle', index=False)

    html_string = f'''
    <html>
      <head><title>HTML Pandas Dataframe with CSS</title></head>
      <link rel="stylesheet" type="text/css" href="df_style.css"/>
      <body>
        {table}
      </body>
    </html>
    '''

    HTML(string=html_string).write_pdf(filename, stylesheets=["df_style.css"])

In [193]:
reg = 0.001

In [194]:
score_df, reg_column = add_reg_column(score_df, reg=reg)

In [195]:
best_clustering_df = get_best_clustering(score_df, reg_column, unique=True)

In [196]:
print_dataframe_to_pdf(best_clustering_df, f'best_clustering_stats_V_score-{reg}*Clusters.pdf')

In [197]:
with PdfPages(f'best_clusterings_V_score-{reg}*Clusters.pdf') as pdf:
    for index, row in best_clustering_df.iterrows():
        p_key = (row['PMID'], row['Level'], 
                 row['bibcoupling'], row['cocitation'], row['citation'], row['text_citation'])
        gt_key = (row['PMID'], row['Level'])
        partition = partitions[p_key]
        gt = ground_truth[gt_key]
        labels_true, labels_pred = align_clusterings_for_sklearn(partition, gt)
        contingency = contingency_matrix(labels_true, labels_pred)
        
        title_str = f"{row['PMID']} - LEVEL {row['Level']}\nReg-Score: {row[reg_column]}"
        fig = plot_contingency_matrix(contingency, title_str)
        pdf.savefig(fig)
        plt.close()
        
        print('.', end='')

.......................................................................................

In [ ]:
plot_catplot(score_df, reg_column, f'full_grid_search_V_score-{reg}*Clusters.png')

## Generate Dummy Results for Testing Visualization

In [5]:
import random
import itertools

from utils.io import get_review_pmids

random.seed(20210302)

In [4]:
PARAM_RANGE = [0, 1, 2]
PARAM_NAMES = ['SIMILARITY_COCITATION',
               'SIMILARITY_BIBLIOGRAPHIC_COUPLING',
               'SIMILARITY_CITATION',
               'SIMILARITY_TEXT_CITATION']
PARAM_GRID = {p: PARAM_RANGE for p in PARAM_NAMES}

In [19]:
def generate_grid_results():
    grid_results = []
    min_value = random.uniform(0.1, 0.3)
    max_value = min_value + random.uniform(0.2, 0.4)
    for param_values in itertools.product(*PARAM_GRID.values()):
        if sum(param_values) == 0:
            continue
        grid_results.append({
            'soi': list(param_values),
            'results': {
                'v_measure_score': random.uniform(min_value, max_value)
            }
        })
    return grid_results

def generate_dummy_results(n_reviews):
    results = []
    for pmid in get_review_pmids()[:n_reviews]:
        results.append({
            'pmid': pmid,
            'level': 1,
            'results': generate_grid_results()
        })
    return results

In [40]:
results = generate_dummy_results(40)

In [68]:
contingency = np.array(
    [
        [8, 2, 5],
        [2, 18, 0]
    ]
)

In [69]:
row_sums = contingency.sum(axis=1)

In [70]:
row_sums

array([15, 20])

In [71]:
contingency_row = contingency / row_sums[:, np.newaxis]

In [72]:
contingency_row

array([[0.53333333, 0.13333333, 0.33333333],
       [0.1       , 0.9       , 0.        ]])

In [73]:
diversity = np.sqrt(np.diag(contingency_row @ contingency_row.T))

In [74]:
diversity

array([0.64291005, 0.90553851])

In [75]:
col_sums = contingency.sum(axis=0)

In [76]:
contingency_col = contingency / col_sums[np.newaxis, :]

In [77]:
contingency_col

array([[0.8, 0.1, 1. ],
       [0.2, 0.9, 0. ]])

In [78]:
purity = np.diag(contingency_row @ contingency_col.T)

In [79]:
purity

array([0.77333333, 0.83      ])

In [80]:
np.mean(purity * diversity)

0.6243903695160768